# SHAP Next-Token Prediction Analysis for Med42 LLM
This notebook demonstrates how to analyze token impacts on next-word prediction using SHAP.

In [ ]:
import sys
sys.path.append('..')

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import shap
import matplotlib.pyplot as plt

# Initialize model in eval mode
model_name = "m42-health/Llama3-Med42-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = (AutoModelForCausalLM
         .from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16)
         .eval()
         .cuda())

# Medical example prompts
prompts = [
    "Patient presents with acute chest pain and elevated troponin levels",
    "MRI reveals a enhancing lesion in the temporal lobe consistent with"
]

In [ ]:
# Initialize and run SHAP analysis
from utils.xai.shap_processor import SHAPProcessor

processor = SHAPProcessor(model, tokenizer)
explanation = processor.explain_text(prompts[0])

# Display results
from IPython.display import HTML
HTML(explanation['html'])

In [ ]:
# Create additional visualizations using plotly
def create_word_impact_plot(words, scores):
    df = pd.DataFrame({
        'Word': words,
        'Impact': scores
    })
    df = df.sort_values('Impact', ascending=True)
    
    fig = px.bar(df, x='Impact', y='Word', orientation='h',
                 title='Word Impact Analysis',
                 color='Impact',
                 color_continuous_scale='RdBu')
    
    fig.update_layout(
        height=max(400, len(words) * 20),
        xaxis_title='Impact Score',
        yaxis_title='Word'
    )
    return fig

# Create and show the plot
fig = create_word_impact_plot(explanation['words'], explanation['scores'])
fig.show()

# Save the plot
fig.write_html("word_impact_analysis.html")

## Batch Processing
The following cell shows how to process multiple prompts and save their visualizations.

In [ ]:
# Create a context manager for memory management
class TorchMemoryManager:
    def __enter__(self):
        torch.cuda.empty_cache()
        gc.collect()
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        torch.cuda.empty_cache()
        gc.collect()

# Modified process_prompts function with memory management
def process_prompts(prompts, batch_size=5):
    results = []
    
    # Process prompts in batches to manage memory
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        with TorchMemoryManager():
            for j, prompt in enumerate(batch):
                idx = i + j
                explanation = processor.explain_text(prompt)
                
                fig = create_word_impact_plot(explanation['words'], explanation['scores'])
                html_path = f"shap_visualization_{idx}.html"
                fig.write_html(html_path)
                
                results.append({
                    'prompt': prompt,
                    'explanation': explanation,
                    'visualization_path': html_path
                })
    
    return results

# Example usage with memory-managed batch processing
prompts = [
    "What are the common symptoms of diabetes?",
    "How is hypertension diagnosed?",
    "What are the risk factors for heart disease?"
]

with TorchMemoryManager():
    results = process_prompts(prompts)